### Read preprocessed data

#### Retrieve Raw Data

In [1]:
import json
import os

def load_jsonl(path):
    row = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            row.append(json.loads(line))
    
    return row

version = 'FullText' # Must be one of: A, AIC, FullText
data_dir = f'../data/processed/SciTLDR-{version}/'

train_raw = load_jsonl(os.path.join(data_dir, 'train.jsonl'))
test_raw = load_jsonl(os.path.join(data_dir, 'test.jsonl'))

print(len(train_raw))
print(len(test_raw))

1992
618


In [2]:
train_raw[0]['target']

'We provide necessary and sufficient analytical forms for the critical points of the square loss functions for various neural networks, and exploit the analytical forms to characterize the landscape properties for the loss functions of these neural networks.'

#### Flatten "Source" Entries from list of 1024 tokens with one output to 1024 inputs and outputs

In [3]:
def flatten(raw):
    flat = []
    for j, ex in enumerate(raw):
        tgt = ex['target']
        chunks = ex['source']
        for i, chunk in enumerate(chunks):
            flat.append({
                'input': chunk,
                'target': tgt,
                'doc_id': j,
                'chunk_id': i
            })
    
    return flat

train_flat = flatten(train_raw)
test_flat = flatten(test_raw)

print(len(train_flat))
print(len(test_flat))

24226
7762


In [4]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    'train': Dataset.from_list(train_flat),
    'test': Dataset.from_list(test_flat)
})

print(dataset)
print(len(dataset['train']['input']))
print(len(dataset['train']['target']))

/home/saharsh/Projects/Paper Summarizer/finetune/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'doc_id', 'chunk_id'],
        num_rows: 24226
    })
    test: Dataset({
        features: ['input', 'target', 'doc_id', 'chunk_id'],
        num_rows: 7762
    })
})
24226
24226


### Fine tune Pegasus Model

In [5]:
from transformers import PegasusTokenizer

model_name = 'google/pegasus-large'
tokenizer = PegasusTokenizer.from_pretrained(model_name)

#### Tokenize the Data

In [6]:
max_input_len = 1024
max_target_len = 128

def preprocess(examples):
    enc = tokenizer(
        examples["input"],
        max_length=max_input_len,
        truncation=True,
        padding=False
    )
    lab = tokenizer(
        text_target=examples["target"],
        max_length=max_target_len,
        truncation=True,
        padding=False
    )
    enc["labels"] = lab["input_ids"]
    
    # keep doc_id/chunk_id if you want custom eval aggregation later
    enc["doc_id"] = examples.get("doc_id", [None]*len(enc["input_ids"]))
    enc["chunk_id"] = examples.get("chunk_id", [None]*len(enc["input_ids"]))
    return enc

tokenized = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names,  # keep only model fields
    desc="Tokenizing",
)

Tokenizing: 100%|██████████| 7762/7762 [00:13<00:00, 578.23 examples/s]


#### Train the Model

In [ ]:
import torch
from transformers import PegasusForConditionalGeneration, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

torch.cuda.empty_cache()

model = PegasusForConditionalGeneration.from_pretrained(model_name)
model.gradient_checkpointing_enable()

args = Seq2SeqTrainingArguments(
    output_dir=f'./pegasus-{version}',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=3,
    predict_with_generate=True,
    generation_max_length=max_target_len,
    generation_num_beams=1,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    report_to='none'
)

collator = DataCollatorForSeq2Seq(tokenizer, model, padding='longest')

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
import numpy as np
import evaluate

rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    return {k: round(v * 100, 2) for k, v in result.items()}


trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_78245/4028849212.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [9]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,2.585500,3.704225,25.330000,6.800000,17.920000,17.920000
2,2.175800,3.913685,24.960000,6.490000,17.660000,17.660000
3,2.016200,3.997631,24.860000,6.430000,17.550000,17.560000


The following generation flags are not valid and may be ignored: ['length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/saharsh/Projects/Paper Summarizer/finetune/.venv/lib/python3.13/site-packages/transformers/modeling_utils.py:4037: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 256, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=4545, training_loss=2.4394391520450873, metrics={'train_runtime': 8605.1021, 'train_samples_per_second': 8.446, 'train_steps_per_second': 0.528, 'total_flos': 2.0933748676942234e+17, 'train_loss': 2.4394391520450873, 'epoch': 3.0})